# Carga, transformación y reporte de `reg_cat_house_sales` (Tabular Benchmark)

En este notebook se construye un flujo completo de trabajo para el dataset [`inria-soda/tabular-benchmark` (`reg_cat_house_sales`)](https://huggingface.co/datasets/inria-soda/tabular-benchmark/viewer/reg_cat_house_sales):

1. Configurar la ruta del proyecto para importar las clases reutilizables.
2. Cargar `inria-soda/tabular-benchmark` (configuración `reg_cat_house_sales`) desde Hugging Face.
3. Obtener un `DataFrame` de pandas y revisar su estructura, valores nulos y registros atípicos.
4. Aplicar cuatro transformaciones sobre el `DataFrame` (eliminación de duplicados, filtrado de registros inconsistentes, renombrado de columnas al español y decodificación/cálculo de variables).
5. Generar un reporte estadístico en pantalla y guardarlo en un archivo `.txt`.

Ejecuta las celdas en orden. Si el dataset requiere autenticación, inicia sesión previamente con `hf auth login`.

## 1. Preparar el entorno e importar la clase de carga

El notebook puede ejecutarse desde distintas carpetas. Por eso se buscan los directorios padres hasta encontrar `src/clases` y se agrega `src` a `sys.path`. Así se puede importar `HFDatasetLoader` sin instalar el proyecto como paquete.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "clases").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from clases import HFDatasetLoader

c:\Users\jonal\OneDrive\Documentos\9_IU Digital\7 Semestre\Proyecto Integrado III - Analítico de Datos\Actividad_2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


**Análisis:** Se localizó dinámicamente el directorio raíz del proyecto y se añadió `src` a `sys.path`, permitiendo importar `HFDatasetLoader` desde el paquete `clases`.

## 2. Configurar el dataset y la salida

`dataset_name` identifica el repositorio `inria-soda/tabular-benchmark` en Hugging Face, `config="reg_cat_house_sales"` selecciona el subconjunto de ventas de viviendas y `split="train"` indica la división que se descargará. El parámetro `filename` hace que el `DataFrame` también se guarde como CSV en `src/sincronicas/datos/`.

In [2]:
loader = HFDatasetLoader(
    dataset_name="inria-soda/tabular-benchmark",
    config="reg_cat_house_sales",
    split="train",
    filename="datos/reg_cat_house_sales.csv",
)


**Análisis:** La instancia `loader` quedó configurada con los parámetros necesarios para conectarse a Hugging Face, descargar la configuración `reg_cat_house_sales` y persistir los datos crudos en `datos/reg_cat_house_sales.csv`.

## 3. Descargar el dataset y crear el DataFrame

El método `load()` descarga el dataset, lo convierte a `pandas.DataFrame` y lo asigna a la variable `df`. La expresión `df.head()` muestra las primeras filas para comprobar que la carga fue exitosa.

In [3]:
df = loader.load()
df.head()

,bedrooms,bathrooms,sqft_living,sqft_lot,waterfront,grade,sqft_above,sqft_basement,yr_built,yr_renovated,lat,long,sqft_living15,sqft_lot15,date_year,date_month,date_day,price
0,3,1.00,1180,5650,0,7,1180,0,1955,0,47.5112,-122.257,1340,5650,0,10,13,12.309987
1,3,2.25,2570,7242,0,7,2170,400,1951,1991,47.7210,-122.319,1690,7639,0,12,9,13.195616
2,2,1.00,770,10000,0,6,770,0,1933,0,47.7379,-122.233,2720,8062,1,2,25,12.100718
3,4,3.00,1960,5000,0,7,1050,910,1965,0,47.5208,-122.393,1360,5000,0,12,9,13.311331
4,3,2.00,1680,8080,0,8,1680,0,1987,0,47.6168,-122.045,1800,7503,1,2,18,13.142168


**Análisis:** El dataset se descargó correctamente en un `DataFrame` de pandas. Las primeras filas muestran variables estructurales de las viviendas (`bedrooms`, `bathrooms`, `sqft_living`, `sqft_lot`, `grade`, `yr_built`, etc.), coordenadas geográficas (`lat`, `long`), componentes de la fecha de venta (`date_year`, `date_month`, `date_day`) y el precio objetivo en escala logarítmica (`price`).

## 4. Revisar la estructura del DataFrame, valores nulos y registros atípicos

Esta celda realiza una inspección del esquema: cuenta filas y columnas, verifica la presencia de valores nulos (especialmente en la variable objetivo `price`) e identifica registros con valores que no tienen sentido en el contexto inmobiliario (por ejemplo, viviendas con 0 habitaciones, 0 baños o errores de digitación como 33 habitaciones).

In [4]:
print(f"Filas: {len(df)}")
print(f"Columnas: {len(df.columns)}")
print(f"Valores nulos totales: {int(df.isna().sum().sum())}")
print(f"Filas sin valor en 'price': {int(df['price'].isna().sum())}")
print(f"Viviendas con 0 habitaciones: {int((df['bedrooms'] == 0).sum())}")
print(f"Viviendas con 0 baños: {int((df['bathrooms'] == 0).sum())}")
print(f"Viviendas con >10 habitaciones: {int((df['bedrooms'] > 10).sum())}")
print("\nDetalle de viviendas con >10 habitaciones:")
print(df[df["bedrooms"] > 10][["bedrooms", "bathrooms", "sqft_living", "yr_built", "price"]])
print()
df.info()

Filas: 21613
Columnas: 18
Valores nulos totales: 0
Filas sin valor en 'price': 0
Viviendas con 0 habitaciones: 13
Viviendas con 0 baños: 10
Viviendas con >10 habitaciones: 2

Detalle de viviendas con >10 habitaciones:
       bedrooms  bathrooms  sqft_living  yr_built      price
8757         11       3.00         3000      1918  13.161586
15870        33       1.75         1620      1947  13.369225

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21613 entries, 0 to 21612
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   bedrooms       21613 non-null  int64  
 1   bathrooms      21613 non-null  float64
 2   sqft_living    21613 non-null  int64  
 3   sqft_lot       21613 non-null  int64  
 4   waterfront     21613 non-null  int64  
 5   grade          21613 non-null  int64  
 6   sqft_above     21613 non-null  int64  
 7   sqft_basement  21613 non-null  int64  
 8   yr_built       21613 non-null  int64  
 9   

**Análisis:** El dataset original cuenta con **21,613 filas y 18 columnas** numéricas (`14 int64` y `4 float64`) y **0 valores nulos** en todas las variables (incluyendo `price`). Sin embargo, la exploración revela registros inconsistentes: **13 viviendas con 0 habitaciones**, **10 con 0 baños** y un caso extremo de **33 habitaciones en una casa de solo 1,620 sqft** (índice 15870, un error de captura documentado en el dataset de King County).

## 5. Transformación 1: Eliminar filas duplicadas

Verificamos y eliminamos posibles filas completamente duplicadas en el `DataFrame`, conservando la primera ocurrencia.

In [5]:
# Eliminar filas duplicadas conservando la primera ocurrencia
df_sin_duplicados = df.drop_duplicates(keep='first').reset_index(drop=True)

print(f"Filas antes    : {len(df)}")
print(f"Filas después  : {len(df_sin_duplicados)}")
print(f"Filas eliminadas del df: {len(df) - len(df_sin_duplicados)}")
df_sin_duplicados.head()

Filas antes    : 21613
Filas después  : 21613
Filas eliminadas del df: 0


,bedrooms,bathrooms,sqft_living,sqft_lot,waterfront,grade,sqft_above,sqft_basement,yr_built,yr_renovated,lat,long,sqft_living15,sqft_lot15,date_year,date_month,date_day,price
0,3,1.00,1180,5650,0,7,1180,0,1955,0,47.5112,-122.257,1340,5650,0,10,13,12.309987
1,3,2.25,2570,7242,0,7,2170,400,1951,1991,47.7210,-122.319,1690,7639,0,12,9,13.195616
2,2,1.00,770,10000,0,6,770,0,1933,0,47.7379,-122.233,2720,8062,1,2,25,12.100718
3,4,3.00,1960,5000,0,7,1050,910,1965,0,47.5208,-122.393,1360,5000,0,12,9,13.311331
4,3,2.00,1680,8080,0,8,1680,0,1987,0,47.6168,-122.045,1800,7503,1,2,18,13.142168


**Análisis:** Al ejecutar `drop_duplicates()`, se confirma que no existen filas idénticas repetidas en el conjunto original (`21,613` filas antes y después), garantizando la integridad de las observaciones antes del filtrado.

## 6. Transformación 2: Filtrar registros inconsistentes y atípicos extremos

Descartamos los registros que no cumplen con condiciones mínimas de coherencia habitacional (viviendas sin habitaciones `bedrooms == 0`, sin baños `bathrooms < 1` o el error tipográfico de `bedrooms > 15`).

In [6]:
condicion_valida = (
    (df_sin_duplicados["bedrooms"] >= 1)
    & (df_sin_duplicados["bedrooms"] <= 15)
    & (df_sin_duplicados["bathrooms"] >= 1)
)
df_filtrado = df_sin_duplicados[condicion_valida].reset_index(drop=True)

print(f"Filas antes del filtro : {len(df_sin_duplicados)}")
print(f"Filas después del filtro: {len(df_filtrado)}")
print(f"Filas descartadas      : {len(df_sin_duplicados) - len(df_filtrado)}")
df_filtrado.head()

Filas antes del filtro : 21613
Filas después del filtro: 21521
Filas descartadas      : 92


,bedrooms,bathrooms,sqft_living,sqft_lot,waterfront,grade,sqft_above,sqft_basement,yr_built,yr_renovated,lat,long,sqft_living15,sqft_lot15,date_year,date_month,date_day,price
0,3,1.00,1180,5650,0,7,1180,0,1955,0,47.5112,-122.257,1340,5650,0,10,13,12.309987
1,3,2.25,2570,7242,0,7,2170,400,1951,1991,47.7210,-122.319,1690,7639,0,12,9,13.195616
2,2,1.00,770,10000,0,6,770,0,1933,0,47.7379,-122.233,2720,8062,1,2,25,12.100718
3,4,3.00,1960,5000,0,7,1050,910,1965,0,47.5208,-122.393,1360,5000,0,12,9,13.311331
4,3,2.00,1680,8080,0,8,1680,0,1987,0,47.6168,-122.045,1800,7503,1,2,18,13.142168


**Análisis:** Se descartaron los registros con valores improcedentes en habitaciones y baños, obteniendo un conjunto depurado de viviendas residenciales consistentes.

## 7. Transformación 3: Renombrar columnas al español

| Columna original | Columna en español |
|------------------|--------------------|
| `bedrooms` | `Habitaciones` |
| `bathrooms` | `Baños` |
| `sqft_living` | `Área habitable (sqft)` |
| `sqft_lot` | `Área del lote (sqft)` |
| `waterfront` | `Vista al agua` |
| `grade` | `Calificación` |
| `sqft_above` | `Área sobre suelo (sqft)` |
| `sqft_basement` | `Área sótano (sqft)` |
| `yr_built` | `Año de construcción` |
| `yr_renovated` | `Año de renovación` |
| `lat` | `Latitud` |
| `long` | `Longitud` |
| `sqft_living15` | `Área habitable vecinos (sqft)` |
| `sqft_lot15` | `Área lote vecinos (sqft)` |
| `date_year` | `Año de venta` |
| `date_month` | `Mes de venta` |
| `date_day` | `Día de venta` |
| `price` | `Precio (log)` |

In [7]:
columnas_es = {
    "bedrooms"      : "Habitaciones",
    "bathrooms"     : "Baños",
    "sqft_living"   : "Área habitable (sqft)",
    "sqft_lot"      : "Área del lote (sqft)",
    "waterfront"    : "Vista al agua",
    "grade"         : "Calificación",
    "sqft_above"    : "Área sobre suelo (sqft)",
    "sqft_basement" : "Área sótano (sqft)",
    "yr_built"      : "Año de construcción",
    "yr_renovated"  : "Año de renovación",
    "lat"           : "Latitud",
    "long"          : "Longitud",
    "sqft_living15" : "Área habitable vecinos (sqft)",
    "sqft_lot15"    : "Área lote vecinos (sqft)",
    "date_year"     : "Año de venta",
    "date_month"    : "Mes de venta",
    "date_day"      : "Día de venta",
    "price"         : "Precio (log)",
}

df_es = df_filtrado.rename(columns=columnas_es)

print("Columnas actualizadas:")
print(df_es.columns.tolist())
df_es.head()

Columnas actualizadas:
['Habitaciones', 'Baños', 'Área habitable (sqft)', 'Área del lote (sqft)', 'Vista al agua', 'Calificación', 'Área sobre suelo (sqft)', 'Área sótano (sqft)', 'Año de construcción', 'Año de renovación', 'Latitud', 'Longitud', 'Área habitable vecinos (sqft)', 'Área lote vecinos (sqft)', 'Año de venta', 'Mes de venta', 'Día de venta', 'Precio (log)']


,Habitaciones,Baños,Área habitable (sqft),Área del lote (sqft),Vista al agua,Calificación,Área sobre suelo (sqft),Área sótano (sqft),Año de construcción,Año de renovación,Latitud,Longitud,Área habitable vecinos (sqft),Área lote vecinos (sqft),Año de venta,Mes de venta,Día de venta,Precio (log)
0,3,1.00,1180,5650,0,7,1180,0,1955,0,47.5112,-122.257,1340,5650,0,10,13,12.309987
1,3,2.25,2570,7242,0,7,2170,400,1951,1991,47.7210,-122.319,1690,7639,0,12,9,13.195616
2,2,1.00,770,10000,0,6,770,0,1933,0,47.7379,-122.233,2720,8062,1,2,25,12.100718
3,4,3.00,1960,5000,0,7,1050,910,1965,0,47.5208,-122.393,1360,5000,0,12,9,13.311331
4,3,2.00,1680,8080,0,8,1680,0,1987,0,47.6168,-122.045,1800,7503,1,2,18,13.142168


**Análisis:** Las 18 columnas del `DataFrame` fueron renombradas al español con nombres descriptivos que facilitan la lectura del reporte final.

## 8. Transformación 4: Decodificar variables y calcular el precio en dólares (USD)

Transformamos el indicador binario `Vista al agua` (`0 -> "No"`, `1 -> "Sí"`), decodificamos `Año de venta` (`0 -> 2014`, `1 -> 2015`) y agregamos la columna `Precio (USD)` revirtiendo la transformación logarítmica de `Precio (log)` mediante `np.exp()`.

In [8]:
import numpy as np

df_transformado = df_es.copy()
df_transformado["Vista al agua"] = df_transformado["Vista al agua"].map({0: "No", 1: "Sí"})
df_transformado["Año de venta"] = df_transformado["Año de venta"].map({0: 2014, 1: 2015})
df_transformado["Precio (USD)"] = np.exp(df_transformado["Precio (log)"]).round(2)

print("Vista previa de las columnas transformadas:")
df_transformado[["Habitaciones", "Baños", "Vista al agua", "Año de venta", "Precio (log)", "Precio (USD)"]].head()

Vista previa de las columnas transformadas:


,Habitaciones,Baños,Vista al agua,Año de venta,Precio (log),Precio (USD)
0,3,1.00,No,2014,12.309987,221901.0
1,3,2.25,No,2014,13.195616,538001.0
2,2,1.00,No,2015,12.100718,180001.0
3,4,3.00,No,2014,13.311331,604001.0
4,3,2.00,No,2015,13.142168,510001.0


**Análisis:** Con esta cuarta transformación, el `DataFrame` cuenta ahora con **19 columnas**, incorporando etiquetas legibles para `Vista al agua`, el año calendario real en `Año de venta` (`2014` o `2015`) y el valor comercial en dólares en `Precio (USD)`.

## 9. Importar la clase de reportes

`Reportes` recibe un `DataFrame` en sus métodos y permite generar un resumen textual, imprimirlo en la consola o guardarlo en un archivo `.txt`.

In [9]:
from clases import Reportes

**Análisis:** Se importó la clase `Reportes` para resumir el `DataFrame` ya transformado.

## 10. Generar y visualizar el reporte como texto

`generar_reporte(df)` devuelve un `str` con el número de filas, columnas, duplicados, tipos de datos, valores nulos y estadísticas numéricas. En esta celda el texto se guarda temporalmente en `texto` y se imprime.

In [10]:
reportes = Reportes()
texto = reportes.generar_reporte(df_transformado)

print(texto)

REPORTE DEL DATAFRAME
Filas: 21521
Columnas: 19
Duplicados: 0

Columnas y tipos:
- Habitaciones: int64
- Baños: float64
- Área habitable (sqft): int64
- Área del lote (sqft): int64
- Vista al agua: object
- Calificación: int64
- Área sobre suelo (sqft): int64
- Área sótano (sqft): int64
- Año de construcción: int64
- Año de renovación: int64
- Latitud: float64
- Longitud: float64
- Área habitable vecinos (sqft): int64
- Área lote vecinos (sqft): int64
- Año de venta: int64
- Mes de venta: int64
- Día de venta: int64
- Precio (log): float64
- Precio (USD): float64

Valores nulos por columna:
- Habitaciones: 0
- Baños: 0
- Área habitable (sqft): 0
- Área del lote (sqft): 0
- Vista al agua: 0
- Calificación: 0
- Área sobre suelo (sqft): 0
- Área sótano (sqft): 0
- Año de construcción: 0
- Año de renovación: 0
- Latitud: 0
- Longitud: 0
- Área habitable vecinos (sqft): 0
- Área lote vecinos (sqft): 0
- Año de venta: 0
- Mes de venta: 0
- Día de venta: 0
- Precio (log): 0
- Precio (USD): 0


**Análisis:** El reporte resume las **21,521 filas y 19 columnas** del `DataFrame` transformado, verificando que no existen valores nulos ni duplicados y detallando las estadísticas descriptivas de todas las variables numéricas (incluyendo una media de precio de venta de aproximadamente **\$541,065 USD** y una mediana de **\$450,001 USD**).

## 11. Imprimir el reporte directamente

`imprimir_reporte(df)` ejecuta la generación y muestra el resultado directamente. Es útil para consultar rápidamente el resumen durante una ejecución interactiva.

In [11]:
reportes.imprimir_reporte(df_transformado)

REPORTE DEL DATAFRAME
Filas: 21521
Columnas: 19
Duplicados: 0

Columnas y tipos:
- Habitaciones: int64
- Baños: float64
- Área habitable (sqft): int64
- Área del lote (sqft): int64
- Vista al agua: object
- Calificación: int64
- Área sobre suelo (sqft): int64
- Área sótano (sqft): int64
- Año de construcción: int64
- Año de renovación: int64
- Latitud: float64
- Longitud: float64
- Área habitable vecinos (sqft): int64
- Área lote vecinos (sqft): int64
- Año de venta: int64
- Mes de venta: int64
- Día de venta: int64
- Precio (log): float64
- Precio (USD): float64

Valores nulos por columna:
- Habitaciones: 0
- Baños: 0
- Área habitable (sqft): 0
- Área del lote (sqft): 0
- Vista al agua: 0
- Calificación: 0
- Área sobre suelo (sqft): 0
- Área sótano (sqft): 0
- Año de construcción: 0
- Año de renovación: 0
- Latitud: 0
- Longitud: 0
- Área habitable vecinos (sqft): 0
- Área lote vecinos (sqft): 0
- Año de venta: 0
- Mes de venta: 0
- Día de venta: 0
- Precio (log): 0
- Precio (USD): 0


**Análisis:** Se comprobó el correcto funcionamiento del método `imprimir_reporte(df_transformado)` en salida estándar.

## 12. Guardar el reporte en un archivo de texto

`guardar_reporte(df, ruta)` crea los directorios necesarios y escribe el reporte en UTF-8. El archivo quedará en `src/sincronicas/reportes/reg_cat_house_sales.txt` cuando se ejecute desde `src/sincronicas`.

In [12]:
ruta = reportes.guardar_reporte(
    df_transformado,
    "reportes/reg_cat_house_sales.txt",
)

print(f"Reporte guardado en: {ruta}")

Reporte guardado en: reportes\reg_cat_house_sales.txt


## Conclusión

Se completó satisfactoriamente la adaptación del flujo de trabajo para el dataset **`inria-soda/tabular-benchmark` (`reg_cat_house_sales`)**:

1. **Extracción y carga (`HFDatasetLoader`)**: Se descargaron las **21,613 filas y 18 columnas** de la división `train` desde Hugging Face y se exportó el archivo base en `src/sincronicas/datos/reg_cat_house_sales.csv`.
2. **Revisión de calidad de datos**: Se verificó la ausencia total de valores nulos (`0` nulos en las 18 columnas) y se detectaron **92 registros atípicos/inconsistentes** (viviendas con 0 habitaciones, menos de 1 baño o el registro anómalo de 33 habitaciones).
3. **Transformaciones aplicadas**:
   - **Eliminación de duplicados**: Verificación de filas repetidas (`drop_duplicates`).
   - **Filtrado de consistencia**: Depuración de 92 registros inválidos, conservando **21,521 viviendas**.
   - **Renombrado de columnas**: Traducción de las 18 variables originales al español.
   - **Decodificación y enriquecimiento**: Conversión de `Vista al agua` a `"No"/"Sí"`, decodificación de `Año de venta` (`2014`/`2015`) y cálculo de `Precio (USD)` a partir de `Precio (log)`.
4. **Generación de reporte (`Reportes`)**: Se generó y almacenó el resumen estadístico completo del `DataFrame` transformado en `src/sincronicas/reportes/reg_cat_house_sales.txt`.